# 06. Retriever

Retriever는 Vector Store를 **LangChain 체인에 연결 가능한 형태**로 감싼 인터페이스입니다.  
`as_retriever()`로 변환하면 LCEL 체인의 구성 요소로 사용할 수 있습니다.

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 벡터 스토어 준비
loader = TextLoader("data/ai_basic.txt", encoding="utf-8")
docs = loader.load()
chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50).split_documents(docs)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

print("준비 완료")

C:\Users\user\AppData\Local\Temp\ipykernel_21716\4070562666.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


준비 완료


## 1. 기본 Retriever 생성

In [2]:
# k: 반환할 문서 수
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke("RAG란 무엇인가요?")

print(f"검색된 문서 수: {len(results)}")
for i, doc in enumerate(results):
    print(f"\n[{i+1}] {doc.page_content[:150]}")


검색된 문서 수: 3

[1] RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시네이션 감소, 출처 제공이 가능합니다.
RAG 파이프라인은 문서 로딩 

[2] 대규모 언어 모델(LLM)이란?
대규모 언어 모델(Large Language Model, LLM)은 방대한 텍스트 데이터로 학습된 트랜스포머 기반 모델입니다. GPT-4, Claude, Gemini 등이 대표적입니다. 텍스트 생성, 번역, 요약, 질의응답 등 다양한 언

[3] LangChain이란?
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공합니다. Python과 JavaScript 버전이 있으며, RAG, 에이전트, 챗봇 등 다


## 2. 검색 타입 설정

| search_type | 설명 |
|---|---|
| `similarity` | 코사인 유사도 기반 (기본값) |
| `mmr` | 다양성을 고려한 검색 (중복 최소화) |
| `similarity_score_threshold` | 유사도 임계값 이상만 반환 |

In [3]:
# MMR - 유사하면서도 다양한 문서 검색
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)

results = mmr_retriever.invoke("RAG란 무엇인가요?")
for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.page_content[:150]}")
    print("-"*30)



[1] RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시네이션 감소, 출처 제공이 가능합니다.
RAG 파이프라인은 문서 로딩 
------------------------------
[2] LangChain이란?
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공합니다. Python과 JavaScript 버전이 있으며, RAG, 에이전트, 챗봇 등 다
------------------------------
[3] 합성곱 신경망(CNN, Convolutional Neural Network)은 이미지 처리에 특화된 신경망 구조입니다. 필터를 사용해 이미지의 특징을 추출하며, 이미지 분류, 객체 탐지에 사용됩니다.
순환 신경망(RNN, Recurrent Neural Network)은
------------------------------


In [4]:
# score_threshold - 유사도 임계값 설정
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.5, "k": 5}
)

results = threshold_retriever.invoke("RAG란 무엇인가요?")
print(f"임계값 이상 문서 수: {len(results)}")
for doc in results:
    print("-", doc.page_content[:100])



c:\Users\user\langgraph_modular_rag\.venv\Lib\site-packages\langchain_core\vectorstores\base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='ef5bd84b-e056-4c1a-a2a7-4f1568f8865a', metadata={'source': 'data/ai_basic.txt'}, page_content='RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시네이션 감소, 출처 제공이 가능합니다.\nRAG 파이프라인은 문서 로딩 → 텍스트 분할 → 임베딩 → 벡터 스토어 저장 → 검색 → 답변 생성 순서로 진행됩니다.\n임베딩(Embedding)이란?'), np.float32(0.33177722)), (Document(id='de6bc8d9-af11-4822-9dc1-b94b341fa150', metadata={'source': 'data/ai_basic.txt'}, page_content='대규모 언어 모델(LLM)이란?\n대규모 언어 모델(Large Language Model, LLM)은 방대한 텍스트 데이터로 학습된 트랜스포머 기반 모델입니다. GPT-4, Claude, Gemini 등이 대표적입니다. 텍스트 생성, 번역, 요약, 질의응답 등 다양한 언어 작업을 수행할 수 있습니다.\nLLM의 한계로는 학습 데이터 이후의 최신 정보를 모른다는 점, 할루시네이션(Hallucination) 현상으로 사실과 다른 내용을 생성할 수 있다는 점이 있습니다.\nRAG(Retrieval-Augmented Generation)이란?'), np.float32(0.07074726)), (Document(id='bcf889ee-fc14-4e0f-9d06-e6

임계값 이상 문서 수: 0


## 3. Retriever를 LCEL 체인에 연결

In [7]:
from langchain_core.runnables import RunnableLambda

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 검색 결과를 하나의 문자열로 합치는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# retriever → format 체인
retrieve_chain = retriever | RunnableLambda(format_docs)

context = retrieve_chain.invoke("임베딩이란?")
# print(context)
# print(len(context))
print(type(context))


<class 'str'>


In [10]:
# 리스트의 요소를 하나의 문자열로 만들기 >> 프롬프트에서 리스트를 문자열로 전달할 때 자주 사용
words = ["aaa","bbb","ccc"]
# "/n/n".join(words)
",".join(words)  # 구분자로 열결 한다.

'aaa,bbb,ccc'

## 4. MultiQueryRetriever 맛보기

- 단일 쿼리를 여러 쿼리로 확장해 더 풍부한 검색 결과를 가져옴(Advanced RAG에서 자세히 다룹니다)  
- 하나의 입력 질문에 LLM이 여러 관점의 검색 질문을 바꿔서 검색하고 결과를 가져옴

- langchain.retrievers 모듈을 쓰려면 langchain-classic을 설치해야함.  
    ```uv add langchain-classic```

In [11]:
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
base_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# base_retriever
# llm이 사용자 질문에 대해서 여러 관점에서 검색을 수행하도록 객체를 생성함

multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
     # include_original=True # 원래 질문을 검색에 포함시킬 경우, 기본은 False
)

# # llm이 여러 관점에서 질문을 만들어서, 벡터 DB 유사도 검색 수행,  문서를 검색하고, 각
results = multi_retriever.invoke("AI의 학습 방법을 알려줘")
print(f"검색된 문서 수: {len(results)}")
for doc in results[:3]:
    print("-", doc.page_content[:120])

검색된 문서 수: 4
- 인공지능(AI)의 기초
인공지능이란 무엇인가?
인공지능(Artificial Intelligence, AI)은 컴퓨터 시스템이 인간의 지능적 행동을 모방하도록 하는 기술입니다. 학습, 추론, 문제 해결, 언어 이해 등
- 강화학습(Reinforcement Learning)은 에이전트가 환경과 상호작용하면서 보상을 최대화하는 방향으로 학습합니다. 게임 AI, 로봇 제어 등에 활용됩니다.
딥러닝이란?
딥러닝(Deep Learning)은 
- 지도학습(Supervised Learning)은 입력과 정답 레이블이 쌍으로 주어진 데이터를 학습합니다. 분류(Classification)와 회귀(Regression) 문제에 사용됩니다. 대표적인 알고리즘으로는 선형


## 정리

- `as_retriever()`로 VectorStore를 LCEL 체인에 연결합니다.
- `similarity`, `mmr`, `similarity_score_threshold` 검색 방식 선택 가능
- Retriever는 `invoke(query)` → `List[Document]` 반환
